In [ ]:
import pandas as pd
import anthropic # or anthropic for Claude
from pathlib import Path
import json

class SurveyAnalyzer:
    def __init__(self, api_key, model="gpt-4"):
        """Initialize with your AI API key"""
        self.client = openai.OpenAI(api_key=api_key)
        self.model = model
        
    def load_spreadsheets(self, file_paths):
        """Load multiple spreadsheets into a combined dataframe"""
        dfs = []
        for path in file_paths:
            if path.endswith('.xlsx'):
                df = pd.read_excel(path)
            elif path.endswith('.csv'):
                df = pd.read_csv(path)
            dfs.append(df)
        
        # Combine all dataframes
        combined_df = pd.concat(dfs, ignore_index=True)
        return combined_df
    
    def analyze_question_responses(self, df, question_column):
        """Analyze all responses for a specific question"""
        responses = df[question_column].dropna().tolist()
        
        prompt = f"""Analyze these free text responses to a survey question:

{chr(10).join(f"- {resp}" for resp in responses[:100])} # Limit for token management

Please provide:
1. Common themes and patterns
2. Categories that emerge from the responses
3. A set of classification rules that could be used to categorize future responses
4. Key insights and sentiments

Format your response as JSON with keys: themes, categories, rules, insights"""
        
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": "You are a data analysis expert specializing in qualitative research and pattern recognition."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.3
        )
        
        return json.loads(response.choices[0].message.content)
    
    def generate_ruleset(self, df, question_columns):
        """Generate comprehensive ruleset for all questions"""
        ruleset = {}
        
        for question in question_columns:
            print(f"Analyzing: {question}")
            analysis = self.analyze_question_responses(df, question)
            ruleset[question] = analysis
        
        return ruleset
    
    def apply_rules_to_new_data(self, df, question_column, rules):
        """Apply generated rules to categorize new responses"""
        responses = df[question_column].dropna().tolist()
        
        prompt = f"""Using these classification rules:
{json.dumps(rules, indent=2)}

Categorize each of these responses:
{chr(10).join(f"{i+1}. {resp}" for i, resp in enumerate(responses))}

Return a JSON array with objects containing: response_index, category, confidence"""
        
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": "You are a classification expert. Apply the rules precisely."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.1
        )
        
        return json.loads(response.choices[0].message.content)

# Usage example
if __name__ == "__main__":
    # Initialize analyzer
    analyzer = SurveyAnalyzer(api_key="sk-proj-REDACTED")
    
    # Load your spreadsheets
    files = ['../Downloads/250905 Risk Heuristics survey download.xlsx"]
    df = analyzer.load_spreadsheets(files)
    
    # Analyze specific questions
    questions = ["What improvements would you suggest?", "What did you like most?"]
    ruleset = analyzer.generate_ruleset(df, questions)
    
    # Save ruleset
    with open('analysis_rules.json', 'w') as f:
        json.dump(ruleset, f, indent=2)
    
    # Apply rules to new data
    categorized = analyzer.apply_rules_to_new_data(
        df, 
        "What improvements would you suggest?", 
        ruleset["What improvements would you suggest?"]["rules"]
    )
    
    print(json.dumps(categorized, indent=2))


SyntaxError: unterminated string literal (detected at line 93) (648751726.py, line 93)